# 3. Controller Initialization (Expanded)

Initializes timing, control gains (`kp`, `kd`), and joint targets. This defines the behavior of the PD controller.

---

```python
class LeftArmRaise:
    def __init__(self):
        self.dt = 0.02
        self.time_ = 0.0

        self.stage_duration = 3.0

        self.kp = 60.0
        self.kd = 1.5

        self.low_cmd = unitree_hg_msg_dds__LowCmd_()
        self.low_state = None
        self.first_update = False

        self.crc = CRC()
        self.done = False

        self.current_stage = -1

        self.joints = [
            G1JointIndex.LeftShoulderPitch,
            G1JointIndex.LeftShoulderRoll,
            G1JointIndex.LeftShoulderYaw,
            G1JointIndex.LeftElbow,
            G1JointIndex.LeftWristRoll,
            G1JointIndex.LeftWristPitch,
            G1JointIndex.LeftWristYaw,
        ]

        self.target_raise = [-0.3, 0.2, 0.0, -1.0, 0.0, 0.5, 0.0]
        self.target_extend = [-0.3, 0.2, 0.0, -1.0, 0.0, 1.0, 0.2]

        self.initial_pose = None
```

---

## 🧠 Big Picture: What This Section Defines

This constructor defines:

> 🎯 **The dynamics, timing, and behavior of the robot arm**

It sets:

* How fast the controller runs
* How stiff or compliant the robot feels
* What motion the robot will execute
* How the controller progresses through time

---

## ⏱️ Control Timing

### `self.dt = 0.02`

```python
self.dt = 0.02
```

This defines the **control loop timestep**:

```text
dt = 0.02 seconds → 50 Hz control loop
```

---

### Why 50 Hz?

* Fast enough for smooth motion
* Slow enough for Python + DDS stability
* Matches typical humanoid control rates

---

### ⚠️ Engineering Insight

From real systems:

* Too slow → jerky motion
* Too fast → CPU/network overload

---

### `self.time_ = 0.0`

Tracks **elapsed time** inside the controller:

```python
self.time_ += self.dt
```

This allows:

* Time-based motion
* Stage transitions
* Trajectory scheduling

---

## 🎬 Motion Scheduling

### `self.stage_duration = 3.0`

Each stage lasts:

```text
3 seconds
```

So total motion:

| Stage | Time    |
| ----- | ------- |
| 1     | 0–3 s   |
| 2     | 3–6 s   |
| 3     | 6–9 s   |
| 4     | 9–15 s  |
| 5     | 15–18 s |

---

### 🧠 Why This Matters

This creates a **finite-state machine based on time**:

```text
time → determines behavior
```

---

## ⚙️ Control Gains (THE MOST IMPORTANT PART)

### `self.kp = 60.0`

```python
self.kp = 60.0
```

This is the **stiffness** of the joint.

---

### Interpretation:

```text
High kp → stiff, precise
Low kp  → soft, compliant
```

---

### From engineering perspective:

* Acts like a **spring constant**
* Higher values:

  * Faster response
  * Higher torque
  * Higher risk (gear stress)

---

### ⚠️ Real Robot Insight

From the technical report:

> High stiffness increases precision but can cause gear damage during impacts.

---

### `self.kd = 1.5`

```python
self.kd = 1.5
```

This is the **damping coefficient**.

---

### Interpretation:

```text
kd resists motion → prevents oscillation
```

---

### Analogy:

* `kp` → spring
* `kd` → shock absorber

---

### Without `kd`:

* Overshoot
* Oscillations
* Unstable motion

---

## 🧱 Command and State Structures

### `self.low_cmd`

```python
self.low_cmd = unitree_hg_msg_dds__LowCmd_()
```

This is the **command buffer** sent to the robot.

It contains:

* Target positions (`q`)
* Velocities (`dq`)
* Gains (`kp`, `kd`)
* Torque (`tau`)

---

### `self.low_state`

```python
self.low_state = None
```

This will later store:

```python
msg.motor_state[j].q
```

→ the **current joint positions**

---

### `self.first_update`

```python
self.first_update = False
```

Used to ensure:

> We **do not move the robot** until we know its current pose

---

### 🧠 Why This Is Critical

Without this:

* Robot might jump to wrong position
* Dangerous motion

---

## 🔐 Safety and Control Flags

### `self.crc`

```python
self.crc = CRC()
```

Used to:

* Validate command messages
* Ensure integrity before sending

---

### `self.done`

```python
self.done = False
```

Tracks whether motion is complete.

Used in main loop to:

```python
if ctrl.done:
    sys.exit(0)
```

---

### `self.current_stage`

```python
self.current_stage = -1
```

Used for:

* Tracking stage transitions
* Avoiding repeated print messages

---

## 🦾 Joint Selection

```python
self.joints = [...]
```

This defines the **control subset**:

* Only left arm joints
* Ordered for kinematic consistency

---

### 🧠 Key Insight

You are creating:

> A **logical kinematic chain on top of hardware indices**

---

## 🎯 Target Trajectories

### `self.target_raise`

```python
[-0.3, 0.2, 0.0, -1.0, 0.0, 0.5, 0.0]
```

Represents:

> The pose for **raising the arm**

---

### `self.target_extend`

```python
[-0.3, 0.2, 0.0, -1.0, 0.0, 1.0, 0.2]
```

Represents:

> The pose for **extending the wrist**

---

### ⚠️ Important Concept

These are:

```text
Joint-space targets (not Cartesian)
```

Meaning:

* You are controlling **angles**, not positions in space

---

## 🧠 Initial Pose

### `self.initial_pose = None`

This will later store:

```python
[msg.motor_state[j].q for j in self.joints]
```

---

### Why This Matters

Instead of assuming a starting pose:

> You dynamically capture the robot’s real pose

This ensures:

* Smooth motion
* No discontinuities
* Safe operation

---

## 🔬 Engineering Insight (From Real Robots)

This section defines a **PD-controlled dynamical system**:

```text
state (q, dq)
   ↓
error = target - state
   ↓
PD controller
   ↓
torque
   ↓
motion
```

---

## 🤖 Connection to Reinforcement Learning

This is where RL plugs in.

Right now:

```python
cmd.q = predefined_target
```

In RL:

```python
cmd.q = policy(state)
```

---

### Key Insight

This constructor defines:

> The **action space + dynamics interface** for RL

---

## 🚀 Summary

This section defines:

| Component   | Role                 |
| ----------- | -------------------- |
| `dt`        | Control frequency    |
| `kp`, `kd`  | Motion behavior      |
| `joints`    | Controlled DOFs      |
| `target_*`  | Desired trajectories |
| `low_cmd`   | Action buffer        |
| `low_state` | State buffer         |
| `time_`     | Motion scheduling    |

---

> 🔥 This is the **heart of the controller**—everything that follows depends on how this is configured.


